# FINANCE 384 Assignment 1 – Part A

## A.5 Out-of-Sample Prediction Performance

This notebook evaluates the predictive performance of:

- the **selected Random Forest** from A.4; and
- the **pooled OLS benchmark** from A.2.

Required A.5 metrics:

1. pooled RMSE;
2. mean monthly Spearman rank correlation;
3. number of months;
4. number of stock-month rows.

Random Forest is evaluated on the **validation and test periods**. Pooled OLS is evaluated on the **test period only**. Both models are compared on exactly the same test stock-month rows.

The selected Random Forest specification from A.4 is:

- `n_estimators = 100`
- `max_depth = 2`
- `min_samples_leaf = 8`
- `max_features = 0.3`

In the final merged assignment notebook, the selected Random Forest object should be carried forward directly from A.4. This standalone A.5 notebook reconstructs the same fixed-seed, training-only selected specification solely so the section can be run independently.


### A.5 evaluation protocol

- **Random Forest validation:** January 2015–December 2018
- **Random Forest test:** January 2019–November 2022
- **Pooled OLS test:** January 2019–November 2022
- **OLS estimation sample:** training + validation
- **Random Forest estimation sample:** training only
- **Test comparison:** identical stock-month rows for both models

Lower RMSE indicates smaller forecast error. Higher mean monthly Spearman indicates better cross-sectional ranking of stocks by next-month realised excess return.


In [ ]:
# A.5.1 Imports and fixed settings

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

PANEL_FILE = "FINANCE384_assignmentA_development_panel.csv"
DICTIONARY_FILE = "FINANCE384_stock_month_data_dictionary.csv"

TARGET = "target_ret_excess_tp1"
RANDOM_SEED = 384

RF_PARAMS = {
    "n_estimators": 100,
    "max_depth": 2,
    "min_samples_leaf": 8,
    "max_features": 0.3,
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
}

RF_PARAMS


In [ ]:
# A.5.2 Load supplied data

panel = pd.read_csv(PANEL_FILE)
data_dictionary = pd.read_csv(DICTIONARY_FILE)

panel["date"] = pd.to_datetime(panel["date"])

print("Panel shape:", panel.shape)
print(
    "Date range:",
    panel["date"].min().date(),
    "to",
    panel["date"].max().date(),
)
print("Unique stocks:", panel["permno"].nunique())
print(
    "Duplicate stock-month rows:",
    panel.duplicated(["permno", "date"]).sum(),
)


In [ ]:
# A.5.3 Define the common revised A.1 predictor information

numeric_predictors = [
    "size",
    "bm",
    "mom12_2",
    "vol12",
    "beta60",
    "ivol60",
    "turnover",
    "dollar_volume",
    "amihud_illiq",
    "divyield",
    "gross_profit",
    "roe",
    "asset_growth",
    "leverage",
    "accruals",
    "mkt_12m",
    "mkt_vol_12m",
    "down_market",
]

continuous_predictors = [
    col for col in numeric_predictors
    if col != "down_market"
]

binary_predictors = ["down_market"]
categorical_predictors = ["ff49_code"]

print("Numeric predictors:", len(numeric_predictors))


### Revised A.1 target construction

The target \(r^e_{i,t+1}\) is attached using a calendar-month stock join so that the realised return used for evaluation is always from the immediately following calendar month.


In [ ]:
# A.5.4 Construct next-calendar-month excess-return target

panel = (
    panel
    .sort_values(["permno", "date"])
    .reset_index(drop=True)
)

panel["month"] = panel["date"].dt.to_period("M")

next_month_return = (
    panel[
        ["permno", "month", "ret_excess_t"]
    ]
    .rename(columns={"ret_excess_t": TARGET})
    .assign(month=lambda df: df["month"] - 1)
)

panel = panel.merge(
    next_month_return,
    on=["permno", "month"],
    how="left",
    validate="one_to_one",
)

print(
    "Rows with valid next-month target:",
    panel[TARGET].notna().sum(),
)


### Revised A.1 missing-data treatment

Missing continuous characteristics are treated using contemporaneous cross-sectional information:

1. create a missingness indicator before imputation;
2. fill missing values using the same-month FF49 industry median;
3. use the same-month market median as fallback.

The missingness indicators are retained as common model inputs.


In [ ]:
# A.5.5 Create missingness indicators and impute characteristics

missing_characteristics = [
    col
    for col in continuous_predictors
    if panel[col].isna().any()
]

missing_indicator_columns = []

for col in missing_characteristics:
    indicator = f"{col}_was_missing"

    panel[indicator] = (
        panel[col]
        .isna()
        .astype(int)
    )

    missing_indicator_columns.append(indicator)

    industry_month_median = (
        panel
        .groupby(["month", "ff49_code"])[col]
        .transform("median")
    )

    market_month_median = (
        panel
        .groupby("month")[col]
        .transform("median")
    )

    panel[col] = (
        panel[col]
        .fillna(industry_month_median)
        .fillna(market_month_median)
    )

print(
    "Missingness indicators created:",
    len(missing_indicator_columns),
)

print(
    "Remaining missing continuous values:",
    int(
        panel[continuous_predictors]
        .isna()
        .sum()
        .sum()
    ),
)


In [ ]:
# A.5.6 Apply the prescribed chronological split

analysis = panel.loc[
    panel[TARGET].notna()
].copy()

train = analysis.loc[
    (analysis["date"] >= "1990-01-01")
    & (analysis["date"] <= "2014-12-31")
].copy()

validation = analysis.loc[
    (analysis["date"] >= "2015-01-01")
    & (analysis["date"] <= "2018-12-31")
].copy()

test = analysis.loc[
    (analysis["date"] >= "2019-01-01")
    & (analysis["date"] <= "2022-11-30")
].copy()

train_valid = (
    pd.concat(
        [train, validation],
        axis=0,
    )
    .sort_values(["date", "permno"])
    .reset_index(drop=True)
)

sample_summary = pd.DataFrame({
    "Sample": [
        "Training",
        "Validation",
        "OLS estimation (Train + Validation)",
        "Test",
    ],
    "Months": [
        train["month"].nunique(),
        validation["month"].nunique(),
        train_valid["month"].nunique(),
        test["month"].nunique(),
    ],
    "Stock-month rows": [
        len(train),
        len(validation),
        len(train_valid),
        len(test),
    ],
})

sample_summary


In [ ]:
# A.5.7 Define final common base predictors

raw_feature_columns = (
    continuous_predictors
    + binary_predictors
    + missing_indicator_columns
    + categorical_predictors
)

print(
    "Raw predictor columns:",
    len(raw_feature_columns),
)


### Common preprocessing

The same preprocessing framework used in A.1–A.4 is retained:

- continuous predictors are standardised using training-sample parameters;
- `down_market` and missingness indicators are passed through unchanged;
- FF49 industry membership is one-hot encoded;
- preprocessing is fitted on training data only and applied unchanged to validation, training + validation, and test samples.


In [ ]:
# A.5.8 Fit common preprocessing on training only

preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            StandardScaler(),
            continuous_predictors,
        ),
        (
            "binary",
            "passthrough",
            binary_predictors
            + missing_indicator_columns,
        ),
        (
            "industry",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_predictors,
        ),
    ],
    remainder="drop",
)

X_train_raw = train[
    raw_feature_columns
].copy()

X_validation_raw = validation[
    raw_feature_columns
].copy()

X_train_valid_raw = train_valid[
    raw_feature_columns
].copy()

X_test_raw = test[
    raw_feature_columns
].copy()

y_train = train[TARGET].to_numpy()
y_validation = validation[TARGET].to_numpy()
y_train_valid = train_valid[TARGET].to_numpy()
y_test = test[TARGET].to_numpy()

preprocessor.fit(
    X_train_raw
)

X_train = preprocessor.transform(
    X_train_raw
)

X_validation = preprocessor.transform(
    X_validation_raw
)

X_train_valid = preprocessor.transform(
    X_train_valid_raw
)

X_test = preprocessor.transform(
    X_test_raw
)

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_validation.shape)
print("OLS train+validation matrix:", X_train_valid.shape)
print("Test matrix:", X_test.shape)


## Reconstruct the assessed models

For this standalone notebook:

- pooled OLS is estimated on **training + validation**, as required;
- the selected Random Forest is reconstructed using the exact A.4-winning specification and is fitted on **training only**.

In the final merged notebook, the Random Forest object retained from A.4 should be used directly rather than fitted again.


In [ ]:
# A.5.9 Fit pooled OLS on training + validation

ols_model = LinearRegression(
    fit_intercept=True
)

ols_model.fit(
    X_train_valid,
    y_train_valid,
)

ols_test_pred = ols_model.predict(
    X_test
)

print(
    "OLS estimation observations:",
    len(y_train_valid),
)

print(
    "OLS test predictions:",
    len(ols_test_pred),
)


In [ ]:
# A.5.10 Reconstruct the A.4-selected Random Forest on training only

selected_random_forest = RandomForestRegressor(
    **RF_PARAMS
)

selected_random_forest.fit(
    X_train,
    y_train,
)

rf_validation_pred = (
    selected_random_forest.predict(
        X_validation
    )
)

rf_test_pred = (
    selected_random_forest.predict(
        X_test
    )
)

print(
    "RF training observations:",
    len(y_train),
)

print(
    "RF validation predictions:",
    len(rf_validation_pred),
)

print(
    "RF test predictions:",
    len(rf_test_pred),
)


## Required A.5 prediction metrics

### Pooled RMSE

Pooled RMSE is calculated across all valid stock-month observations in the relevant period.

### Mean monthly Spearman rank correlation

For each month, Spearman correlation is calculated between predicted and realised next-month excess returns across stocks. The reported metric is the arithmetic mean of these monthly correlations.

Pandas/SciPy's Spearman implementation uses average ranks for tied observations.


In [ ]:
# A.5.11 Evaluation functions

def pooled_rmse(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    return float(
        np.sqrt(
            mean_squared_error(
                actual,
                predicted,
            )
        )
    )


def monthly_spearman_table(
    df,
    actual_col,
    predicted_col,
):
    monthly = (
        df
        .groupby("date")
        .apply(
            lambda month: month[
                predicted_col
            ].corr(
                month[actual_col],
                method="spearman",
            ),
            include_groups=False,
        )
        .rename("spearman")
        .reset_index()
    )

    return monthly


def evaluate_predictions(
    df,
    actual_col,
    predicted_col,
):
    valid = (
        df[
            ["date", actual_col, predicted_col]
        ]
        .dropna()
        .copy()
    )

    monthly_spearman = (
        monthly_spearman_table(
            valid,
            actual_col,
            predicted_col,
        )
    )

    return {
        "rmse": pooled_rmse(
            valid[actual_col],
            valid[predicted_col],
        ),
        "mean_monthly_spearman": float(
            monthly_spearman[
                "spearman"
            ].mean()
        ),
        "months": int(
            monthly_spearman[
                "date"
            ].nunique()
        ),
        "stock_month_rows": int(
            len(valid)
        ),
        "monthly_spearman_table":
            monthly_spearman,
    }


In [ ]:
# A.5.12 Assemble prediction datasets

rf_validation_predictions = validation[
    ["date", "permno", "ticker", TARGET]
].copy()

rf_validation_predictions[
    "rf_pred_excess_return_tp1"
] = rf_validation_pred


test_predictions = test[
    ["date", "permno", "ticker", TARGET]
].copy()

test_predictions[
    "rf_pred_excess_return_tp1"
] = rf_test_pred

test_predictions[
    "ols_pred_excess_return_tp1"
] = ols_test_pred

print(
    "Common test rows:",
    len(test_predictions),
)

print(
    "Missing RF test predictions:",
    int(
        test_predictions[
            "rf_pred_excess_return_tp1"
        ].isna().sum()
    ),
)

print(
    "Missing OLS test predictions:",
    int(
        test_predictions[
            "ols_pred_excess_return_tp1"
        ].isna().sum()
    ),
)


In [ ]:
# A.5.13 Verify identical test rows for both models

common_test_mask = (
    test_predictions[TARGET].notna()
    & test_predictions[
        "rf_pred_excess_return_tp1"
    ].notna()
    & test_predictions[
        "ols_pred_excess_return_tp1"
    ].notna()
)

common_test = (
    test_predictions
    .loc[common_test_mask]
    .copy()
)

print(
    "Original test rows:",
    len(test_predictions),
)

print(
    "Common valid test rows:",
    len(common_test),
)

print(
    "Rows dropped from common comparison:",
    len(test_predictions) - len(common_test),
)

print(
    "Common test months:",
    common_test["date"].dt.to_period("M").nunique(),
)


In [ ]:
# A.5.14 Calculate required A.5 metrics

rf_validation_metrics = evaluate_predictions(
    rf_validation_predictions,
    TARGET,
    "rf_pred_excess_return_tp1",
)

rf_test_metrics = evaluate_predictions(
    common_test,
    TARGET,
    "rf_pred_excess_return_tp1",
)

ols_test_metrics = evaluate_predictions(
    common_test,
    TARGET,
    "ols_pred_excess_return_tp1",
)

a5_results = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Period": "Validation",
        "Pooled RMSE":
            rf_validation_metrics["rmse"],
        "Mean monthly Spearman":
            rf_validation_metrics[
                "mean_monthly_spearman"
            ],
        "Months":
            rf_validation_metrics["months"],
        "Stock-month rows":
            rf_validation_metrics[
                "stock_month_rows"
            ],
    },
    {
        "Model": "Random Forest",
        "Period": "Test",
        "Pooled RMSE":
            rf_test_metrics["rmse"],
        "Mean monthly Spearman":
            rf_test_metrics[
                "mean_monthly_spearman"
            ],
        "Months":
            rf_test_metrics["months"],
        "Stock-month rows":
            rf_test_metrics[
                "stock_month_rows"
            ],
    },
    {
        "Model": "Pooled OLS",
        "Period": "Test",
        "Pooled RMSE":
            ols_test_metrics["rmse"],
        "Mean monthly Spearman":
            ols_test_metrics[
                "mean_monthly_spearman"
            ],
        "Months":
            ols_test_metrics["months"],
        "Stock-month rows":
            ols_test_metrics[
                "stock_month_rows"
            ],
    },
])

a5_results


### Direct test comparison

The required test comparison uses exactly the same stock-month observations for Random Forest and pooled OLS.

Lower pooled RMSE is preferred for forecast accuracy. Higher mean monthly Spearman is preferred for cross-sectional stock ranking.


In [ ]:
# A.5.15 Direct Random Forest vs OLS test comparison

test_comparison = pd.DataFrame({
    "Comparison": [
        "RF test RMSE minus OLS test RMSE",
        "RF RMSE improvement vs OLS (%)",
        "RF test Spearman minus OLS test Spearman",
        "RF validation RMSE minus RF test RMSE",
        "RF validation Spearman minus RF test Spearman",
    ],
    "Value": [
        rf_test_metrics["rmse"]
        - ols_test_metrics["rmse"],

        100
        * (
            ols_test_metrics["rmse"]
            - rf_test_metrics["rmse"]
        )
        / ols_test_metrics["rmse"],

        rf_test_metrics[
            "mean_monthly_spearman"
        ]
        - ols_test_metrics[
            "mean_monthly_spearman"
        ],

        rf_validation_metrics["rmse"]
        - rf_test_metrics["rmse"],

        rf_validation_metrics[
            "mean_monthly_spearman"
        ]
        - rf_test_metrics[
            "mean_monthly_spearman"
        ],
    ],
})

test_comparison


In [ ]:
# A.5.16 Retain monthly Spearman series for audit

rf_validation_monthly_spearman = (
    rf_validation_metrics[
        "monthly_spearman_table"
    ]
    .rename(
        columns={
            "spearman":
                "rf_validation_spearman"
        }
    )
)

rf_test_monthly_spearman = (
    rf_test_metrics[
        "monthly_spearman_table"
    ]
    .rename(
        columns={
            "spearman":
                "rf_test_spearman"
        }
    )
)

ols_test_monthly_spearman = (
    ols_test_metrics[
        "monthly_spearman_table"
    ]
    .rename(
        columns={
            "spearman":
                "ols_test_spearman"
        }
    )
)

test_monthly_rank_comparison = (
    rf_test_monthly_spearman
    .merge(
        ols_test_monthly_spearman,
        on="date",
        how="inner",
    )
)

test_monthly_rank_comparison.head()


In [ ]:
# A.5.17 Final protocol audit

print("A.5 PREDICTION PERFORMANCE AUDIT")
print("-" * 55)

print(
    "Random Forest validation months:",
    rf_validation_metrics["months"],
)

print(
    "Random Forest validation rows:",
    rf_validation_metrics[
        "stock_month_rows"
    ],
)

print(
    "Random Forest test months:",
    rf_test_metrics["months"],
)

print(
    "Random Forest test rows:",
    rf_test_metrics[
        "stock_month_rows"
    ],
)

print(
    "OLS test months:",
    ols_test_metrics["months"],
)

print(
    "OLS test rows:",
    ols_test_metrics[
        "stock_month_rows"
    ],
)

print(
    "\nSame test rows used for RF and OLS:",
    (
        rf_test_metrics[
            "stock_month_rows"
        ]
        == ols_test_metrics[
            "stock_month_rows"
        ]
        == len(common_test)
    ),
)

print(
    "Expected test months:",
    47,
)

print(
    "Test period:",
    "Jan 2019 – Nov 2022",
)


## A.5 Summary

A.5 evaluates the selected Random Forest on both validation and test periods and compares it with pooled OLS on the common test sample.

The required outputs are:

- pooled RMSE;
- mean monthly Spearman rank correlation;
- number of months;
- number of stock-month rows.

The Random Forest validation metrics provide evidence on the model-selection period, while the common test comparison provides the untouched out-of-sample evidence needed to assess whether the richer model improves prediction relative to pooled OLS.

These statistical prediction results should be interpreted jointly with the economic portfolio evidence developed in A.6.
